In [1]:
from huggingface_hub import login

login()

In [2]:
from datasets import load_dataset
from pathlib import Path

dataset = load_dataset(path="dvilasuero/finepersonas-v0.1-tiny", split="train")

Path("data").mkdir(parents=True, exist_ok=True)
for i, persona in enumerate(dataset):
    with open(Path("data") / f"persona_{i}.txt", "w") as f:
        f.write(persona["persona"])

/home/nerfeddoge/miniconda3/envs/hf_agent_course/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loading and embedding documents

In [3]:
from llama_index.core import SimpleDirectoryReader

reader = SimpleDirectoryReader(input_dir="data")
documents = reader.load_data()

In [4]:
from llama_index.core import Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline

# Create the pipeline with transformations
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_overlap=0),
        HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5"),
    ]
)

nodes = await pipeline.arun(documents=[Document.example()])
nodes

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13412.39it/s]


[TextNode(id_='3822fc2f-dee8-44f0-8e16-01e3fe5c7002', embedding=[-0.07779841125011444, -0.029093919321894646, 0.002216680906713009, -0.010391009971499443, 0.09002884477376938, -0.0691441148519516, 0.008252838626503944, -0.009724902920424938, 0.01978115923702717, -0.04347209632396698, 0.02659624256193638, -0.00920415110886097, 0.07875444740056992, 0.002549141412600875, 0.042597606778144836, 0.03183040767908096, 0.003503652522340417, 0.020021991804242134, -0.020237060263752937, -0.009460018947720528, 0.04732125252485275, 0.015019438229501247, -2.0764662622241303e-05, 0.0051340945065021515, 0.01222595851868391, 0.09635484218597412, -0.03063942678272724, -0.03588945046067238, -0.021210428327322006, -0.16182997822761536, 0.034438539296388626, -0.022890999913215637, 0.06059261038899422, 0.0316377617418766, -0.022752495482563972, 0.0048812879249453545, -0.011713869869709015, 0.0013522305525839329, -0.03216586261987686, 0.04718421772122383, 0.006247262936085463, 0.01199401542544365, -0.0250942

## Storing and indexing documents

In [5]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore

db = chromadb.PersistentClient(path="./alfred_chroma_db")
chroma_collection = db.get_or_create_collection("alfred")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(),
        HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5"),
    ],
    vector_store=vector_store
)
nodes = await pipeline.arun(documents=documents[:10])
len(nodes)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12704.98it/s]


10

In [6]:
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
index = VectorStoreIndex.from_vector_store(vector_store, embed_model=embed_model)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14315.77it/s]


## Querying a VectorStoreIndex with prompts and LLMs

In [7]:
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI

llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")
query_engine = index.as_query_engine(
    llm=llm,
    response_mode="tree_summarize",
)
query_engine.query("Whats the meaning of life?")

Response(response="The meaning of life is a profound question that varies widely across different cultures, histories, and societies. For someone deeply immersed in Cypriot culture, the meaning of life might be intertwined with the island's rich heritage, its community values, and the resilience of its people through centuries of change. It could also reflect personal philosophies shaped by the Mediterranean lifestyle, emphasizing family, tradition, and the importance of living in harmony with nature and one's surroundings.", source_nodes=[NodeWithScore(node=TextNode(id_='62aaea32-272e-43c8-aede-564c46390f6c', embedding=None, metadata={'file_path': '/home/nerfeddoge/Documents/Personal/AI Agent course - Hugging face/Unit 2.2 - LlamaIndex/data/persona_1.txt', 'file_name': 'persona_1.txt', 'file_type': 'text/plain', 'file_size': 266, 'creation_date': '2026-10-25', 'last_modified_date': '2026-10-25'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'la

## Response Processing

In [8]:
from llama_index.core.evaluation import FaithfulnessEvaluator

evaluator = FaithfulnessEvaluator(llm=llm)
response = query_engine.query(
    "What battles took place in New York City in the American Revolution?"
)

eval_result = evaluator.evaluate_response(response=response)
eval_result.passing

False

In [9]:
from dotenv import load_dotenv
load_dotenv("../.env")
import os
import llama_index

# llama_index.core.set_global_handler(
#     "arize_phoenix",
#     endpoint="https://llamatrace.com/v1/traces"
# )
from phoenix.otel import register

tracer_provider = register(
  project_name="demo_llama_index_hf_agent_course",
  auto_instrument=True
)

/home/nerfeddoge/miniconda3/envs/hf_agent_course/lib/python3.14/site-packages/phoenix/otel/otel.py:449: UserWarning: Could not infer collector endpoint protocol, defaulting to HTTP.
  warnings.warn("Could not infer collector endpoint protocol, defaulting to HTTP.")


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: demo_llama_index_hf_agent_course
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: https://app.phoenix.arize.com/s/nguyenhoangnhat5atv/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {'authorization': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [10]:
response = query_engine.query(
    "What is the name of the someone that is interested in AI and techhnology?"
)
response

Response(response='The provided information does not mention anyone specifically interested in AI and technology. It describes an anthropologist or cultural expert focused on Cypriot culture, history, and society. There is no reference to AI or technology in the given context.', source_nodes=[NodeWithScore(node=TextNode(id_='c5744c88-5570-400b-8bdc-2081cb5c3a17', embedding=None, metadata={'file_path': '/home/nerfeddoge/Documents/Personal/AI Agent course - Hugging face/Unit 2.2 - LlamaIndex/data/persona_1.txt', 'file_name': 'persona_1.txt', 'file_type': 'text/plain', 'file_size': 266, 'creation_date': '2026-10-25', 'last_modified_date': '2026-10-25'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='d22a8d1d-